In [64]:
# Load CSEC2017 synthetic + KD2017 datasets 
from datasets import load_from_disk
import datasets
datasets.disable_caching()
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu") 
num_classes = 9 
KAs = {"0": "miscellaneous (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

from utils.load_data import preprocess
# KDs 
KD_dataset = datasets.load_dataset("csv",data_files={"train": "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/train_data.csv"}, split='train')
KD_dataset = KD_dataset.remove_columns('KSAT ID')
KD_dataset = KD_dataset.map(preprocess)
print(KD_dataset)

from utils.load_data import preprocess_csec, preprocess_csec8
# CSEC2017 specific! CHANGED: train_CSEC2017b to train_CSEC2017c to include class 0 
csec_dataset = datasets.load_dataset("csv",data_files={"train": "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/train_CSEC2017c.csv"}, split='train')
csec_dataset = csec_dataset.remove_columns('Statement Description')
csec_dataset = csec_dataset.remove_columns('label')
csec_dataset = csec_dataset.select_columns(['topics','0','1','2','3','4','5','6','7','8'])
csec_dataset = csec_dataset.map(preprocess_csec)
print(csec_dataset)

Map:   0%|          | 0/576 [00:00<?, ? examples/s]

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 576
})


Map:   0%|          | 0/2143 [00:00<?, ? examples/s]

Dataset({
    features: ['topics', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2143
})


In [65]:
# clean datasets and convert to pandas 
import pandas as pd 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from utils.load_data import clean_text

feature_type = 'tf-idf' 

pandas_dataCSEC = pd.DataFrame({'topics': csec_dataset['topics'], 'labels': csec_dataset['labels']})
pandas_dataCSEC['cleaned'] = pandas_dataCSEC['topics'].apply(clean_text)

# remove knowledge of (indiscriminative)
def remove_knowledge_of(example):
    example = example[13:]
    return example

pandas_dataKD = pd.DataFrame({'topics': KD_dataset['Statement Description'], 'labels': KD_dataset['labels']})
pandas_dataKD['cleaned'] = pandas_dataKD['topics'].apply(clean_text).apply(remove_knowledge_of)
# merge fine-tuning datasets 
mergeds = pd.concat([pd.DataFrame({'topics': pandas_dataCSEC['cleaned'], 'labels': pandas_dataCSEC['labels']}), 
                     pd.DataFrame({'topics': pandas_dataKD['cleaned'], 'labels': pandas_dataKD['labels']})]).reset_index() # for
# convert labels to int 
#mergeds['labels'] = mergeds['labels'].apply(lambda x: [int(i) for i in x])

In [66]:
import evaluate
import numpy as np
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def sigmoid(x):
   return 1/(1 + np.exp(-x))

def compute_metrics(eval_pred):
   predictions, labels = eval_pred
   predictions = sigmoid(predictions)
   predictions = (predictions > 0.5).astype(int).reshape(-1)
   return clf_metrics.compute(predictions=predictions, references=labels.astype(int).reshape(-1))

In [74]:
# Load model 
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from transformers import set_seed 
set_seed(42) 

training_args = TrainingArguments(
        output_dir="ROBERTA",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        logging_steps=100,
        # I added these three 
        eval_strategy="epoch",
        save_strategy="epoch",
        #load_best_model_at_end=True,
        #metric_for_best_model='f1',
        save_total_limit=1,
        dataloader_num_workers=0, # explicitly set to single GPU
    )

def init_model(training_args): 
    model_name='FacebookAI/roberta-base'#'albert-base-v2' 
    model = AutoModelForSequenceClassification.from_pretrained(model_name,
            problem_type="multi_label_classification",
            num_labels=num_classes) #, dropout=0.2) only for DistilBERT
    tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    return model, tokenizer, data_collator


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [75]:
# To do: Train 5 times and average the resulting metrics. 
# To do: 5 different random seeds as well. 
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score 
from datasets import Dataset
import torch
import warnings 
warnings.filterwarnings('ignore')

def tokenize(example): 
    example = tokenizer(example['topics'])                          # tokenize text 
   # example['labels'] = labels       # format labels as torch tenensor 
    return example 
    
n_repeats = 5 # number of seeds 
n_splits = 10 # number of k splits 
precisions = []
recalls = [] 
f1s = [] 
accs = []
for _ in range(n_repeats): 
    kf = KFold(n_splits=n_splits,  shuffle=True) 
    precision = 0 
    recall = 0 
    f1 = 0 
    acc = 0 
    for i, (train, test) in enumerate(kf.split(mergeds)):  # split into k-folds
        print('split ',i)
        # initiate a new model for each split
        model, tokenizer, data_collator = init_model(training_args)
        # extract train and test dataset for k-folds cross-validation
        train_ds = mergeds.loc[train] 
        test_ds = mergeds.loc[test]
        train_dataset = Dataset.from_pandas(train_ds)
        train_dataset = train_dataset.map(tokenize)
        test_dataset = Dataset.from_pandas(test_ds)
        test_dataset = test_dataset.map(tokenize)
        # train 
        trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=test_dataset,
                tokenizer=tokenizer,
                data_collator=data_collator,
                compute_metrics=compute_metrics,)
        trainer.train()
        # compute metrics 
        predictions, og_labels, metrics= trainer.predict(test_dataset)
        predictions = sigmoid(predictions)
        predicted_labels = (predictions > 0.5).astype(int)
        report = classification_report(test_ds['labels'].tolist(),predicted_labels
                                               ,output_dict=True, zero_division=0)
        # gather and normalize metrics by the length of the test dataset 
        print(report['macro avg']) # this is the score we use for the others as well 
        #print(report['micro avg']) # this is what is reported now 
        accuracy = accuracy_score(test_ds['labels'].tolist(), predicted_labels)
        acc += (accuracy * len(test_ds)) / len(mergeds)
        precision += (report['macro avg']['precision'] * len(test_ds)) / len(mergeds)
        recall += (report['macro avg']['recall'] * len(test_ds)) / len(mergeds)
        f1 += (report['macro avg']['f1-score'] * len(test_ds)) / len(mergeds)
    
    precisions.append(precision) 
    recalls.append(recall) 
    f1s.append(f1)
    accs.append(acc) 
# print the results
print('precision: ',np.mean(precisions))
print('precision std: ',np.std(precisions))
print('recall: ',np.mean(recalls))
print('recall std: ',np.std(recalls))
print('f1: ',np.mean(f1s))
print('f1 std: ',np.std(f1s))
print('accuracy: ',np.mean(accs))
print('accuracy std: ',np.std(accs))

split  0


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/2448 [00:00<?, ? examples/s]

Map:   0%|          | 0/271 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.390987,0.847888,0.000000,0.000000,0.000000
2,No log,0.326645,0.868799,0.422383,0.639344,0.315364
3,0.380900,0.299731,0.883149,0.533552,0.679167,0.439353
4,0.380900,0.280851,0.889709,0.576378,0.693182,0.493261
5,0.380900,0.287748,0.892989,0.592824,0.703704,0.512129
6,0.247700,0.275414,0.897089,0.628148,0.697368,0.571429
7,0.247700,0.271875,0.900369,0.635682,0.716216,0.571429
8,0.179400,0.272165,0.899549,0.640235,0.703226,0.587601
9,0.179400,0.269360,0.902419,0.646884,0.719472,0.587601
10,0.179400,0.270785,0.899139,0.637168,0.703583,0.582210


{'precision': 0.7009778403595608, 'recall': 0.561581778437151, 'f1-score': 0.610144854150589, 'support': 371.0}
precision:  0.6944169475299312
precision std:  0.0009751051883458041
recall:  0.58479023921801
recall std:  0.0005034372633921703
f1:  0.626925749562255
f1 std:  0.0007182318364948871
accuracy:  0.5104817947774918
accuracy std:  0.0


In [ ]:
# RoBERTa
precision:  0.6944169475299312
precision std:  0.0009751051883458041
recall:  0.58479023921801
recall std:  0.0005034372633921703
f1:  0.626925749562255
f1 std:  0.0007182318364948871
accuracy:  0.5104817947774918
accuracy std:  0.0

In [ ]:
# ALBERT 
precision:  0.6756647519314891
precision std:  0.0010497058916063117
recall:  0.4842812454545552
recall std:  0.002007643795144332
f1:  0.5492132453945544
f1 std:  0.0017907135092215755
accuracy:  0.42317028319235017
accuracy std:  0.002648032364839992

In [ ]:
BERT
precision:  0.7096383485667477
precision std:  0.00043191777787359164
recall:  0.5871193011879087
recall std:  0.0011479391693208995
f1:  0.6355136715242098
f1 std:  0.0005693183502191168
accuracy:  0.5150422949613829
accuracy std:  0.00044133872747336155

In [63]:
# distilbert output

split  0


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2447 [00:00<?, ? examples/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


...

Map:   0%|          | 0/2448 [00:00<?, ? examples/s]

Map:   0%|          | 0/271 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.399787,0.847888,0.000000,0.000000,0.000000
2,No log,0.337358,0.868389,0.346232,0.708333,0.229111
3,0.397900,0.297700,0.886019,0.501792,0.748663,0.377358
4,0.397900,0.276055,0.899139,0.595395,0.763713,0.487871
5,0.397900,0.277323,0.895039,0.581699,0.738589,0.479784
6,0.253000,0.266440,0.899139,0.621538,0.724014,0.544474
7,0.253000,0.261743,0.904879,0.648485,0.740484,0.576819
8,0.180400,0.260443,0.899549,0.635958,0.708609,0.576819
9,0.180400,0.255247,0.906929,0.655539,0.750000,0.582210
10,0.180400,0.256126,0.909389,0.664643,0.760417,0.590296


{'precision': 0.7615672084981548, 'recall': 0.5730500295733608, 'f1-score': 0.643884030359681, 'support': 371.0}
precision:  0.7052346044206985
precision std:  8.881256810435722e-05
recall:  0.5699055246562277
recall std:  0.0004256870950527958
f1:  0.6216423122103872
f1 std:  0.0003854291372716201
accuracy:  0.497020963589555
accuracy std:  0.0002942258183155744
